# Verificacion de imagen captura con la camara

In [7]:
import subprocess
from pathlib import Path

conda = Path.home() / "miniconda3-ai-studio" / "Scripts" / "conda.exe"

result = subprocess.run(
    [
        str(conda), "create", "-y", "-n", "ai-studio-py38",
        "--override-channels", "-c", "conda-forge",
        "python=3.8", "numpy=1.24.4", "matplotlib", "opencv", "dlib",
        "pillow=10.2", "requests", "tqdm", "ipykernel",
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)

if result.returncode:
    print("Conda terminó con código:", result.returncode)

Retrieving notices: ...working... done
Channels:
 - conda-forge
Platform: win-64
Solving environment: ...working... done

## Package Plan ##

  environment location: C:\Users\rober\miniconda3-ai-studio\envs\ai-studio-py38

  added / updated specs:
    - dlib
    - ipykernel
    - matplotlib
    - numpy=1.24.4
    - opencv
    - pillow=10.2
    - python=3.8
    - requests
    - tqdm


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    aom-3.9.1                  |       he0c23c2_0         1.9 MB  conda-forge
    asttokens-3.0.0            |     pyhd8ed1ab_0          27 KB  conda-forge
    backcall-0.2.0             |     pyh9f0ad1d_0          13 KB  conda-forge
    brotli-1.1.0               |       hcfcfb64_1          19 KB  conda-forge
    brotli-bin-1.1.0           |       hcfcfb64_1          20 KB  conda-forge
    brotli-python-1.1.0        |   py38hd3f51b4_1         314 KB  conda-forge
 

In [9]:
import subprocess
from pathlib import Path

conda = Path.home() / "miniconda3-ai-studio" / "Scripts" / "conda.exe"
env = "ai-studio-py38"

commands = [
    [
        str(conda), "run", "-n", env, "python", "-m", "pip",
        "install", "torch==2.2.2", "torchvision==0.17.2",
        "--index-url", "https://download.pytorch.org/whl/cpu",
    ],
    [
        str(conda), "run", "-n", env, "python", "-m", "pip",
        "install", "facenet-pytorch==2.6.0",
    ],
    [
        str(conda), "run", "-n", env, "python", "-m", "ipykernel",
        "install", "--user", "--name", env,
        "--display-name", "Python 3.8 (AI Studio)",
    ],
]

for command in commands:
    result = subprocess.run(command, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        raise RuntimeError(f"Falló el comando: {' '.join(command)}")

print("Listo. Selecciona el kernel Python 3.8 (AI Studio).")

Looking in indexes: https://download.pytorch.org/whl/cpu
     ---------------------------------------- 0.0/200.8 MB ? eta -:--:--
     ---------------------------------------- 1.6/200.8 MB 9.3 MB/s eta 0:00:22
      -------------------------------------- 3.9/200.8 MB 10.2 MB/s eta 0:00:20
     - ------------------------------------- 6.3/200.8 MB 10.2 MB/s eta 0:00:20
     - -------------------------------------- 7.9/200.8 MB 9.9 MB/s eta 0:00:20
     - -------------------------------------- 9.2/200.8 MB 9.2 MB/s eta 0:00:21
     -- ------------------------------------ 10.7/200.8 MB 8.7 MB/s eta 0:00:22
     -- ------------------------------------ 12.1/200.8 MB 8.4 MB/s eta 0:00:23
     -- ------------------------------------ 13.6/200.8 MB 8.2 MB/s eta 0:00:23
     --- ----------------------------------- 15.5/200.8 MB 8.2 MB/s eta 0:00:23
     --- ----------------------------------- 17.6/200.8 MB 8.5 MB/s eta 0:00:22
     --- ----------------------------------- 19.1/200.8 MB 8.5 MB/s et

In [1]:
# Esto lo tengo yo porque tengo problemas con la librería MKL y me da error de threading (Se me muere el kernel de jupyter)

import os
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"

%matplotlib inline

In [2]:
import cv2
import numpy as np


cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Extraccion de canales RGB
    b_channel = frame.copy()
    b_channel[:, :, 1] = 0  # Set G channel to 0
    b_channel[:, :, 2] = 0  # Set R channel to 0

    g_channel = frame.copy()
    g_channel[:, :, 0] = 0  # Set B channel to 0
    g_channel[:, :, 2] = 0  # Set R channel to 0
    
    r_channel = frame.copy()
    r_channel[:, :, 0] = 0  # Set B channel to 0
    r_channel[:, :, 1] = 0  # Set G channel to 0

    # Concatenar los canales y el original vertical y horizontal
    row_1_concat = cv2.hconcat([frame, b_channel])
    row_2_concat = cv2.hconcat([g_channel, r_channel])
    final_concat = cv2.vconcat([row_1_concat, row_2_concat])
    
    # Mostrar la imagen concatenada
    cv2.imshow('Original', final_concat)

    # Presiona la tecla 'q' o 'esc' para salir del bucle ¡En minuscula!
    key = cv2.waitKey(1) & 0xFF

    if key == ord('q') or key == 27:
        break   

# Release the video capture and close all windows
cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

# Verificacion de que el sistema de aprendizaje profundo pueda detectar un rostro

Inicializacion de los modelos a utilizar

In [1]:
import subprocess
from pathlib import Path

conda = Path.home() / "miniconda3-ai-studio" / "Scripts" / "conda.exe"
env = "ai-studio-py38"

subprocess.check_call([
    str(conda), "install", "-y", "-n", env,
    "--force-reinstall", "--override-channels", "-c", "conda-forge",
    "pillow=10.2.0",
])

subprocess.check_call([
    str(conda), "run", "-n", env, "python", "-c",
    "from PIL import Image; import PIL; print('Pillow:', PIL.__version__); print('Importación correcta')",
])

0

In [4]:
import os
import sys
from pathlib import Path

dll_dir = Path(sys.prefix) / "Library" / "bin"
print("Python del kernel:", sys.executable)
print("Carpeta DLL:", dll_dir)

_dll_handle = os.add_dll_directory(str(dll_dir))  # Mantener esta referencia activa

from PIL import Image
print("Pillow cargó correctamente:", Image.__file__)

from facenet_pytorch import MTCNN, InceptionResnetV1
print("FaceNet cargó correctamente")

Python del kernel: c:\Users\rober\miniconda3-ai-studio\envs\ai-studio-py38\python.exe
Carpeta DLL: c:\Users\rober\miniconda3-ai-studio\envs\ai-studio-py38\Library\bin
Pillow cargó correctamente: c:\Users\rober\miniconda3-ai-studio\envs\ai-studio-py38\lib\site-packages\PIL\Image.py
FaceNet cargó correctamente


In [6]:
import subprocess
from pathlib import Path

conda = Path.home() / "miniconda3-ai-studio" / "Scripts" / "conda.exe"

subprocess.check_call([
    str(conda), "update", "-y", "-n", "ai-studio-py38",
    "--override-channels", "-c", "conda-forge",
    "ca-certificates", "certifi", "openssl",
])

0

In [1]:
from facenet_pytorch import MTCNN, InceptionResnetV1
import torch

# Initialize MTCNN for face detection
mtcnn = MTCNN(image_size=160, margin=0, post_process=False)

# Initialize InceptionResnetV1 for recognition
resnet = InceptionResnetV1(pretrained='vggface2').eval()


c:\Users\rober\miniconda3-ai-studio\envs\UNIE_CNN\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 107M/107M [00:13<00:00, 8.02MB/s] 


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image

# Carga una imagen de ejemplo para probar el recorte de cara, cargala tu. No preguntes como jeje.
img = cv2.imread(<image_path>)  # Reemplaza <image_path> con la ruta de tu imagen

# Test de face cropping
img_cropped = mtcnn(img, save_path='result/cropped.png')
img2show = img_cropped.permute(1, 2, 0).int().numpy()  
img2show = img2show[:, :, ::-1]  

plt.imshow(img2show)

def cropFace(img):
    img_cropped = mtcnn(img, save_path='result/cropped.png')
    if img_cropped is None:
        return None
    img2show = img_cropped.permute(1, 2, 0).int().numpy()  
    img2show = img2show[:, :, ::-1]  
    return img2show.astype(np.uint8)  

SyntaxError: invalid syntax (81030757.py, line 5)

In [ ]:


cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    cropped_face = cropFace(frame)
    size = min(frame.shape[0], frame.shape[1])
    if cropped_face is not None:
        cropped_face_resized = cv2.resize(cropped_face, (size, size))  # Resize to 160x160 for the model

    frame_concatenated = cv2.hconcat([frame, cropped_face_resized])
    cv2.imshow('Pitufo CAM', frame_concatenated)

    # Presiona la tecla 'q' o 'esc' para salir del bucle ¡En minuscula!
    key = cv2.waitKey(1) & 0xFF

    if key == ord('q') or key == 27:
        break

cap.release()
cv2.destroyAllWindows()